# Proyecto Process Mining
Enzo Joaquín Santillán Málaga


## Actividad 1: Data engineering
### 1. ¿Qué columna o combinación de columnas usarías como identificador de caso (sesión, conversación, usuario o usuario más día), y qué pasaría con las variantes si eligieras otra?
El identificador de caso mas conveniente es el número del archivo csv, porque cada archivo es una conversación completa entre bot y usuario. Los datos no contienen usuario ni fecha reales, así que no se puede usar usuario ni usuario+día como identificador. Si se eligiera Falante se tendrían solo 2 casos enormes que mezclan todas las conversaciones. Si se eligiera Tipo o Subtipo de Touchpoint, que son las actividades, cada caso agruparía todas las apariciones de una misma actividad y habría una sola variante por caso. Con Mensagem, cada evento sería su propio caso por ser texto casi único. En todos estos casos las trazas no representarían un recorrido real, y el análisis de variantes no tendría sentido.

### 2. Los mensajes han sido transformados en actividades usando touchpoints ¿Crees que hay otra forma de crear actividades con un nivel de abstracción adecuado (por ejemplo intención del usuario, tipo de respuesta del bot), y qué técnica otra usarías para etiquetarlos?
Sí, se pueden crear actividades de otra forma. Del lado del usuario usaría la intención (consulta de producto, reclamo, solicitud de humano, cierre), etiquetada con un LLM en modo zero-shot sobre una lista cerrada de categorías. Del lado del bot clasificaría el tipo de respuesta (concluyente, con duda, desviada, segura) con el mismo método. El análisis de sentimiento de los mensajes del usuario lo usaría como un atributo del evento y no como actividad, para detectar en qué punto de la conversación el cliente se frustra. Mantendría entre 5 y 10 etiquetas por lado para que las variantes sean interpretables, y validaría una muestra manualmente.

### 3. Si dos mensajes tienen el mismo timestamp o este solo tiene precisión de segundos, ¿cómo garantizas el orden correcto dentro de cada caso? ¿qué regla aplicarías para este caso?
El resolver este caso seria algo simple, el algoritmo se tiene que basar en el orden de quien ha mandado el mensaje. Dentro del csv se puede ver como hay un orden estricto de respuesta bot y respuesta usuario. Por lo que un algoritmo que ordene las actividades segun el orden del orador puede asegurar que no se cometan errores relacionados al timestamp.

### 4. ¿Cómo derivarías un evento de inicio y de fin de conversación si el log no los trae?
Crearía dos eventos sintéticos por caso. Inicio de conversación se coloca antes del primer mensaje, ya que el primer evento de cada caso es el saludo del bot. Fin de conversación se coloca tras el último mensaje, y su causa se infiere así: si el último mensaje se clasifica como cierre o despedida, la conversación terminó con normalidad; si hay una inactividad prolongada o el usuario deja de responder, es un abandono. No exigiría un cierre de ambos lados, porque en los datos 5 de 13 conversaciones terminan en abandono, inactividad, ignorar sugerencias o dificultad, sin despedida. El motivo de fin se guarda como atributo para poder comparar variantes exitosas y fallidas.


## Ejemplo de transformación de los csvs a log

In [ ]:
import re
from pathlib import Path

import pandas as pd
import pm4py
import numpy as np                  # números aleatorios para el log de ejemplo
import matplotlib.pyplot as plt     # gráficos
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 300)

# Funciona tanto si el notebook se ejecuta desde la carpeta del proyecto como desde su carpeta padre
CARPETA = next(p for p in (Path("conversas_250505"), Path("Proyecto_ProcessM/conversas_250505")) if p.is_dir())
FECHA_BASE = pd.Timestamp("2025-05-05")

partes = []
for archivo in CARPETA.glob("DIALOGO*.csv"):
    caso = re.search(r"(\d+)", archivo.stem).group(1)
    df = pd.read_csv(archivo, encoding="utf-8-sig").drop(columns=["Mensagem"])
    df["case:concept:name"] = caso
    df["event_index"] = range(len(df))
    partes.append(df)

df_log = pd.concat(partes, ignore_index=True).rename(columns={
    "Falante": "org:resource",
    "Subtipo de Touchpoint": "concept:name",
    "Tipo de Touchpoint": "fase",
})
df_log["actividad_falante"] = df_log["org:resource"] + " | " + df_log["concept:name"]
# timestamp sintético: solo sirve para ordenar, no para medir duraciones
df_log["time:timestamp"] = FECHA_BASE + pd.to_timedelta(df_log["event_index"], unit="s")
df_log = df_log.sort_values(["case:concept:name", "event_index"],
                            key=lambda s: s.astype(int)).reset_index(drop=True)
df_log = pm4py.format_dataframe(df_log, case_id="case:concept:name",
                                activity_key="concept:name", timestamp_key="time:timestamp")

log = pm4py.convert_to_event_log(df_log)
print(f"Casos: {len(log)} | Eventos: {len(df_log)}")


Casos: 13 | Eventos: 159


## Actividad 2: Análisis del flujo 
### 0. Cargar event_log_en.csv


In [29]:
def cargar_log(ruta, col_caso, col_actividad, col_tiempo, col_recurso=None):
    """Lee un log de eventos (CSV o XES) y estandariza los nombres de las columnas."""
    # 1. Leer el archivo según su extensión
    if ruta.endswith(".csv"):
        tabla = pd.read_csv(ruta)
    elif ruta.endswith(".xes") or ruta.endswith(".xes.gz"):
        import pm4py                      # solo se necesita para leer archivos XES
        tabla = pm4py.read_xes(ruta)
    else:
        raise ValueError("Formato no soportado: use un archivo .csv o .xes")

    # 2. Renombrar las columnas a los nombres estándar
    nombres_nuevos = {col_caso: "caso", col_actividad: "actividad", col_tiempo: "tiempo"}
    if col_recurso is not None and col_recurso in tabla.columns:
        nombres_nuevos[col_recurso] = "recurso"
    tabla = tabla.rename(columns=nombres_nuevos)

    # 3. Convertir el tiempo a fecha y hora, y el caso a texto
    tabla["tiempo"] = pd.to_datetime(tabla["tiempo"], format="%d/%m/%Y %H:%M", utc=True)
    tabla["tiempo"] = tabla["tiempo"].dt.tz_localize(None)
    tabla["caso"] = tabla["caso"].astype(str)

    # 4. Ordenar los eventos de cada caso por tiempo (el orden de llegada puede no ser el cronológico)
    tabla = tabla.sort_values(["caso", "tiempo"], kind="mergesort")
    tabla = tabla.reset_index(drop=True)
    return tabla

RUTA_LOG = "event_log_en.csv"

COL_CASO = "CASE_ID"            
COL_ACTIVIDAD = "ACTIVITY_EN"     
COL_TIEMPO = "START_TIMESTAMP"        
COL_RECURSO = "RECURSO"    

log = cargar_log(RUTA_LOG, COL_CASO, COL_ACTIVIDAD, COL_TIEMPO, COL_RECURSO)
log = log.drop(columns=["Unnamed: 0"])
log.head(10)

,caso,EVENT_ID,ACTIVITY,tiempo,END_TIMESTAMP,recurso,AGENTE,actividad
0,00044a5f68c82574,1,SAUDAÇÃO INICIAL DO CHATBOT,2025-07-11 04:53:00,11/07/2025 04:53,NaN,ai,Initial chatbot greeting
1,00044a5f68c82574,2,RESPOSTA COM INSATISFAÇÃO,2025-07-11 04:53:00,11/07/2025 04:57,NaN,human,Response with dissatisfaction
2,00044a5f68c82574,3,ENCAMINHAMENTO OUTROS CANAIS,2025-07-11 04:57:00,11/07/2025 04:57,cartoes_credito_outros_produtos,ai,Referral to other channels
3,00044a5f68c82574,4,RESPOSTA COM INSATISFAÇÃO,2025-07-11 04:57:00,11/07/2025 05:00,NaN,human,Response with dissatisfaction
4,00044a5f68c82574,5,ENCAMINHAMENTO OUTROS CANAIS,2025-07-11 05:00:00,11/07/2025 05:00,cartoes_credito_outros_produtos,ai,Referral to other channels
5,00044a5f68c82574,6,RESPOSTA COM INSATISFAÇÃO,2025-07-11 05:00:00,11/07/2025 05:03,NaN,human,Response with dissatisfaction
6,00044a5f68c82574,7,ENCAMINHAMENTO OUTROS CANAIS,2025-07-11 05:03:00,11/07/2025 05:04,cartoes_credito_outros_produtos,ai,Referral to other channels
7,00044a5f68c82574,8,ENTREGA DE INFORMAÇÕES,2025-07-11 05:04:00,11/07/2025 05:06,NaN,human,Information delivery
8,00044a5f68c82574,9,ENCAMINHAMENTO OUTROS CANAIS,2025-07-11 05:06:00,11/07/2025 05:07,cartoes_credito_outros_produtos,ai,Referral to other channels
9,00044a5f68c82574,10,REDIRECIONAMENTO A OUTRO CANAL,2025-07-11 05:07:00,11/07/2025 05:09,NaN,human,Redirection to other channel


In [30]:
def revisar_calidad(log):
    """Revisa problemas típicos de calidad de un log y devuelve un resumen."""
    resultados = {}
    resultados["Filas (eventos)"] = len(log)
    resultados["Casos"] = log["caso"].nunique()
    resultados["Actividades distintas"] = log["actividad"].nunique()
    resultados["Casos sin identificador"] = int(log["caso"].isna().sum())
    resultados["Eventos sin actividad"] = int(log["actividad"].isna().sum())
    resultados["Eventos sin fecha"] = int(log["tiempo"].isna().sum())

    # Eventos repetidos exactamente (mismo caso, actividad y momento)
    repetidos = log.duplicated(subset=["caso", "actividad", "tiempo"])
    resultados["Eventos duplicados"] = int(repetidos.sum())

    # Eventos con la misma marca de tiempo que el evento anterior del mismo caso
    diferencias = log.groupby("caso")["tiempo"].diff()
    resultados["Eventos con la misma hora que el anterior"] = int((diferencias == pd.Timedelta(0)).sum())

    # Casos con un solo evento no permiten medir ningún tiempo
    eventos_por_caso = log.groupby("caso").size()
    resultados["Casos con un solo evento"] = int((eventos_por_caso == 1).sum())
    return pd.Series(resultados)

revisar_calidad(log)

Filas (eventos)                              2070
Casos                                         300
Actividades distintas                          33
Casos sin identificador                         0
Eventos sin actividad                           2
Eventos sin fecha                               0
Eventos duplicados                              0
Eventos con la misma hora que el anterior     536
Casos con un solo evento                        0
dtype: int64

In [32]:

antes = len(log)
log = log.dropna(subset=["actividad"]).reset_index(drop=True)
print(f"Eventos eliminados: {antes - len(log)} | Eventos restantes: {len(log)}")


Eventos eliminados: 0 | Eventos restantes: 2068


### 1. Patrones de control del flujo: Identifica en el log tratado un caso de secuencia, uno de elección exclusiva (XOR), uno de paralelismo (AND) y un bucle entre touchpoints. ¿Qué evidencia en las trazas respalda cada uno?
1.- Secuencia: La primera relación que encontramos es al inicio con la secuencia de Initial chatbot greeting y luego suele ir Clarification of intent - Directed flows - Spontaneous thanks.

2.- XOR: Una secuencia de actividades que se ve repetidamente en el log es Response with dissatisfaction (163 veces), a Clarification of intent (56) o a Information delivery (55).

3.- AND: Hay evidencia en el log de posible AND con Delivery of an unresolved response y Solution delivery (9 casos en un orden y 9 en el otro) ambos aparecen en orden intercambiado, pero esto de todas maneras no demuestra que sean una compuerta AND, mas que nada por el timestamp que no son actividades simultaneas. 

4.- Bucle: En este caso se ve un bucle bastante claro que es Response with dissatisfaction <-> Referral to other channels, con 273 transiciones en un sentido y 227 en el otro. 

### 2. Bucles y retrabajo: Algunos touchpoints se repiten dentro de una conversación (reformulaciones del usuario, repreguntas del bot). ¿Cómo distingues un bucle legítimo (mensajes que se refieren al mismo tema) de un fallo del chatbot (el bot hizo repitió la pregunta)? explica tu respuesta, cada respuesta es única para extra pregunta

Con este log, que no incluye el texto de los mensajes, distingo ambos bucles por tres señales. Primera: quién repite. Un usuario que reformula, como en Clarification of intent, es legítimo, mientras que el bot que repite la misma actividad sin cambios, como Referral to other channels, es sospechoso. Segunda: si hay progreso entre repeticiones, por ejemplo Response with data o Solution delivery. Tercera: el desenlace. Entre los 111 casos con 2 o más Referral to other channels, solo 6 terminan en agradecimiento y 87 en queja o insatisfacción, lo que indica fallo del bot. Con el texto de los mensajes, mediría además la similitud entre ellos.

### 3. Variantes: ¿Cuántas variantes hay y qué porcentaje de casos cubren las 5 más frecuentes? ¿Qué decisión tomarías sobre las variantes raras (filtrar, agrupar o conservar)?

In [ ]:
def construir_trazas(log):
    """Crea una tabla con una fila por caso: su traza, su inicio, su fin y su duración en días."""
    filas = []
    for id_caso, eventos_del_caso in log.groupby("caso"):
        lista_actividades = list(eventos_del_caso["actividad"])
        primer_momento = eventos_del_caso["tiempo"].iloc[0]
        ultimo_momento = eventos_del_caso["tiempo"].iloc[-1]
        segundos = (ultimo_momento - primer_momento).total_seconds()

        fila = {
            "caso": id_caso,
            "traza": " > ".join(lista_actividades),
            "primera_actividad": lista_actividades[0],
            "ultima_actividad": lista_actividades[-1],
            "n_eventos": len(lista_actividades),
            "inicio": primer_momento,
            "fin": ultimo_momento,
            "duracion_dias": segundos / 86400,        # 86400 segundos tiene un día
        }
        filas.append(fila)
    return pd.DataFrame(filas)


def tabla_variantes(trazas):
    """Agrupa los casos por traza y calcula cuántos casos tiene cada variante."""
    grupos = trazas.groupby("traza")
    variantes = pd.DataFrame({
        "casos": grupos.size(),
        "duracion_media_dias": grupos["duracion_dias"].mean(),
    }).reset_index()
    variantes = variantes.sort_values("casos", ascending=False).reset_index(drop=True)
    variantes["porcentaje"] = 100 * variantes["casos"] / variantes["casos"].sum()
    variantes["acumulado"] = variantes["porcentaje"].cumsum()
    variantes.insert(0, "variante", range(1, len(variantes) + 1))
    return variantes


def variantes_para_cubrir(variantes, porcentaje_objetivo):
    """Cuenta cuántas variantes (de la más frecuente a la menos frecuente) hacen falta para cubrir un porcentaje de casos."""
    for posicion in range(len(variantes)):
        if variantes["acumulado"].iloc[posicion] >= porcentaje_objetivo:
            return posicion + 1
    return len(variantes)

trazas = construir_trazas(log)
variantes = tabla_variantes(trazas)

print("Casos:", len(trazas), "| Variantes distintas:", len(variantes))
variantes[["variante", "casos", "porcentaje", "acumulado", "duracion_media_dias", "traza"]].round(1)

Casos: 300 | Variantes distintas: 210


,variante,casos,porcentaje,acumulado,duracion_media_dias,traza
0,1,21,7.0,7.0,0.0,Initial chatbot greeting > Response with dissa...
1,2,11,3.7,10.7,0.0,Initial chatbot greeting > Response with dissa...
2,3,11,3.7,14.3,0.0,Initial chatbot greeting > Response with dissa...
3,4,8,2.7,17.0,0.0,Initial chatbot greeting > Response with dissa...
4,5,6,2.0,19.0,0.0,Initial chatbot greeting > Clarification of in...
...,...,...,...,...,...,...
205,206,1,0.3,98.7,0.0,Initial chatbot greeting > Response with dissa...
206,207,1,0.3,99.0,0.0,Initial chatbot greeting > Response with dissa...
207,208,1,0.3,99.3,0.0,Initial chatbot greeting > Solution rejection ...
208,209,1,0.3,99.7,0.0,Initial chatbot greeting > Solution rejection ...


In [ ]:
cantidad = variantes_para_cubrir(variantes, 30)
cantidad1 = variantes_para_cubrir(variantes, 40)
cantidad2 = variantes_para_cubrir(variantes, 50)
cantidad3 = variantes_para_cubrir(variantes, 60)
cantidad4 = variantes_para_cubrir(variantes, 70)
cantidad5 = variantes_para_cubrir(variantes, 80)
print("variantes necesarias para cubrir el 30% de los casos:", cantidad, "de", len(variantes))
print("variantes necesarias para cubrir el 40% de los casos:", cantidad1, "de", len(variantes))
print("variantes necesarias para cubrir el 50% de los casos:", cantidad2, "de", len(variantes))
print("variantes necesarias para cubrir el 60% de los casos:", cantidad3, "de", len(variantes))
print("variantes necesarias para cubrir el 70% de los casos:", cantidad4, "de", len(variantes))
print("variantes necesarias para cubrir el 80% de los casos:", cantidad5, "de", len(variantes))

Variantes con 1 caso: 25
Variantes con 2 casos: 17
Variantes con 3 casos: 12
variantes necesarias para cubrir el 30% de los casos: 14 de 210
variantes necesarias para cubrir el 40% de los casos: 31 de 210
variantes necesarias para cubrir el 50% de los casos: 60 de 210
variantes necesarias para cubrir el 60% de los casos: 90 de 210
variantes necesarias para cubrir el 70% de los casos: 120 de 210
variantes necesarias para cubrir el 80% de los casos: 151 de 210


Los 5 casos más frecuentes comprenden el 19% del total de casos, habiendo un total de 210 variantes las 31 mas grandes ya comprenden el aproximadamente el 40% del total, luego de eso cada aumento de 10% aumenta 30 variantes lo que significa que cada variante es de 0.3% que significa que representa solo 1 caso. Por eso el log conviene que se quede con las 31 variantes mas grandes, para evitar variantes unicas.

In [61]:
# Conservar solo los casos de las 31 variantes más frecuentes
N_VARIANTES = 31
traza_top = variantes["traza"].iloc[:N_VARIANTES]

trazas_filtradas = trazas[trazas["traza"].isin(traza_top)].copy()
casos_validos = set(trazas_filtradas["caso"])
log_filtrado = log[log["caso"].isin(casos_validos)].reset_index(drop=True)
variantes_filtradas = tabla_variantes(trazas_filtradas)

print(f"Casos: {len(trazas)} -> {len(trazas_filtradas)} | Variantes: {len(variantes)} -> {len(variantes_filtradas)}")


Casos: 300 -> 121 | Variantes: 210 -> 31


## Actividad 3: Process discovery
### 1. Descubre el proceso con Alpha miner en pm4py y evalúa las métricas sound del modelo descubierto

### 2. Descubre el proceso con Inductive miner en pm4py y evalúa las métricas sound del modelo descubierto

### 3. Compara cuál de los dos modelos es mejor?